In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

chat_model = init_chat_model(model="openrouter:openai/gpt-5.6-luna", )

In [2]:
from langchain.tools import tool
from pathlib import Path
import subprocess

WORKSPACE = Path("./todo-list-workspace")


@tool(parse_docstring=True)
def list_files(path: str = ".") -> str:
    """
    列出工作区指定目录下的文件和子目录。path 只能是相对路径。

    Args:
        path: 工作区下的相对路径，一定指向目录，默认为.，表示工作区根路径，不能访问工作区外的目录
    """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    if not str(target).startswith(str(workspace_root)):
        return "错误：只允许访问工作区内的目录。"
    if not target.exists():
        return f"错误：目录不存在: {path}"
    if not target.is_dir():
        return f"错误：不是目录: {path}"

    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    if not items:
        return f"目录为空: {path}"
    lines = []

    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")

    return "\n".join(lines)


@tool(parse_docstring=True)
def read_file(path: str) -> str:
    """
    读取工作区中的文本文件内容。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
    """

    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许读取工作区内的文件。"
    if not file_path.exists():
        return f"错误：文件不存在: {path}"

    return file_path.read_text(encoding="utf-8")


@tool(parse_docstring=True)
def write_file(path: str, content: str) -> str:
    """
    写入工作区中的文本文件。path 只能是相对路径。

    Args:
        path: 工作区内的文件名
        content: 写入文件的内容
    """

    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "错误：只允许写入工作区内的文件。"

    file_path.write_text(content, encoding="utf-8")
    return f"已写入文件: {path}"


@tool(parse_docstring=True)
def run_tests() -> str:
    """
    在工作区运行 pytest -q，并返回输出。
    不接收任何参数，返回格式为
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"运行测试失败: {e}"

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.messages import HumanMessage
from rich import print as rprint

# 1. 初始化 Agent
agent = create_agent(
    model=chat_model,
    # write_todos 等工具，TodoListMiddleware 需要配合这些工具使用
    tools=[list_files, read_file, write_file, run_tests],
    # 引入 Todo 列表中间件
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "你是一个代码修复助手。遇到多步骤任务时，先使用 write_todos 制定待办事项；"
        "然后读取文件、修复代码并运行测试。工作全部在工作区下进行。"
    ),
)

# 2. 使用invoke进行同步调用
print("正在执行 Agent 任务...")
final_state = agent.invoke(
    {
        "messages": [
            HumanMessage(content="请测试并修复工作区下 my_add.py 文件中的代码")
        ]
    }
)

rprint(final_state)

正在执行 Agent 任务...


{
    'messages': [
        HumanMessage(
            content='请测试并修复工作区下 my_add.py 文件中的代码',
            additional_kwargs={},
            response_metadata={},
            id='90d460cc-5968-4a72-9865-67c12c94d5ba'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'type': 'reasoning.encrypted',
                        'data': 
'gAAAAABqazRlH4L0EMU7FsMPTdvwrl_1mCrylaGfK6prWM_ZEeVCEm53OiV4fli7XX3URplUed2tDfKxwQaWt13XuBEi4bIi_P4UbsqswZASKw_evJ
S0co5zdsZGH347n7Q3l6p6O4IGwCMc38IBSUtpBm03PrIwxr_T6em6h-fKKCJyRgrC3tt1F-WDBnYgRu8jyfNbl9tehUi3ZJH3-D1lZfXL_WDS1uE5Y
bN-pHisBuu9XNc9XbnABbJrakGNqlETq5LHj7Joo3vccqU4_0gjws9BmLv9PT5AewbRF7zxM66E4k-sifk_v42x-B5ocKM5NNY3-O9UOJJRxblOJyu3
8rq5ApYy0xvJcLSmlsIb4gyBtkumvFVEJiQ-Tq36FR93yWK5BpxIasFwd60xyplaISSlIiAqMg76JwxsXGUtokis_VyHyjFUF78iMgla1DmQNl1ZBoL
3FOKI8fVClyfZ07iEJd1t8SruS_MSOf3uWUkFe2z3NJgI2iTjFt6tlrGn9APCkLEK1akdoUKEcnU1gpi5VbYgYEsrKSQJWjSlyAdMTTOI3Dw79Fk_T-
BRUdxBOlCnhm7JcMwv_XE9cgfazLHtjBt3ALhVwohp5Qyk7bBT850-fAGQbddPVOeeucMclkmAekBMhgzNIDhSetlEDHAu4waoMMM9oTCnbNpRcq7f9
bS-caPyJ-U027eeeOpZqZ5hQXG18IQ4uLj-3zs_pu7a1dWzKrSKGUl9XHQCmNICDk6pAi4BdHYD9qB2doEaUVJyM8v8Tz_kJrmQINYNP33IQxVhC8BX
Y1QFpt2CDkelyLYm9h6RRh3c0icUTsJ1AOXg_HK7Ym5c75Dk_AWJWS_XWB2UcQne69245b_U7VqMuUJ5yHbvrHfud_rwXdrn9aZeeqc-Wm7bccJRDXa
av4-UeErW9PhWuDU3-WattcuSbZj1nTm64_FwPkBS5lFIMfmI2w25M_2YA3rmNMV5XXuNfjvcBw==',
                        'id': 'rs_0ef633afb504da8c016a6b346559b081a3856852dc12c2c245',
                        'format': 'openai-responses-v1',
                        'index': 0.0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785410660-VCd6mmT36SwC9fkJmpk6',
                'created': 1785410660,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fb2c4-a29b-7991-a013-793930bfedb8-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {'content': '检查工作区并读取 my_add.py 及相关测试', 'status': 'in_progress'},
                            {'content': '定位并修复 my_add.py 中的问题', 'status': 'pending'},
                            {'content': '运行测试并确认修复结果', 'status': 'pending'}
                        ]
                    },
                    'id': 'call_EPedpSHkn8VIXI9tfLmOUcl0',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1357,
                'output_tokens': 84,
                'total_tokens': 1441,
                'input_token_details': {'cache_read': 0, 'cache_creation': 1354},
                'output_token_details': {'reasoning': 13}
            }
        ),
        ToolMessage(
            content="Updated todo list to [{'content': '检查工作区并读取 my_add.py 及相关测试', 'status': 
'in_progress'}, {'content': '定位并修复 my_add.py 中的问题', 'status': 'pending'}, {'content': 
'运行测试并确认修复结果', 'status': 'pending'}]",
            name='write_todos',
            id='e0786433-d995-4a46-8d52-8efcc2392632',
            tool_call_id='call_EPedpSHkn8VIXI9tfLmOUcl0'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785410663-MdcVqIrvo3vGTS6Br3iR',
                'created': 1785410663,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fb2c4-b1fe-7442-b087-f5d01e5b402b-0',
            tool_calls=[
  